# HeartMuLa｜音樂生成（Colab T4）

修改歌詞、曲風與秒數，依序執行後直接試聽並下載 MP3。

**相依性：**依照原始 HeartMuLa Colab 版，使用 FFmpeg **5.1.9 shared build**、PyTorch 2.10／TorchCodec 0.10，以及單 GPU `lazy_load`。 第一次安裝 FFmpeg 及模型可能需要較久，建議課前完成。 **請使用獨立的 Colab 執行階段。**

## 1. 安裝 FFmpeg 5.1.9（只需執行一次）

這段沿用原始可重用建置腳本：檢查版本與 shared libraries；如果已安裝且驗證通過就跳過編譯。

In [ ]:
# 1) FFmpeg 5.1.9 shared build
#
# 已存在且健康 -> 重用
# 已存在但壞掉 -> 重建
# ffmpeg.org 失敗 -> 官方 GitHub n5.1.9 fallback
#
# 此版完全移除會造成 exit 141 的 `ffmpeg -version | head`。

import os
import shutil
import subprocess
from pathlib import Path

FFMPEG_PREFIX = Path("/content/ffmpeg-5.1.9-build")

subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
subprocess.run(
    [
        "sudo", "apt-get", "install", "-y", "-qq",
        "build-essential", "pkg-config", "yasm", "nasm",
        "curl", "xz-utils", "libmp3lame-dev",
    ],
    check=True,
)

ffmpeg_script = r'''
set -euo pipefail

PREFIX="/content/ffmpeg-5.1.9-build"
SRC="/content/ffmpeg-5.1.9"
XZ="/content/ffmpeg-5.1.9.tar.xz"
TGZ="/content/ffmpeg-5.1.9-github.tar.gz"

register_runtime() {
    export PATH="$PREFIX/bin:$PATH"
    export LD_LIBRARY_PATH="$PREFIX/lib:${LD_LIBRARY_PATH:-}"
    echo "$PREFIX/lib" | sudo tee /etc/ld.so.conf.d/ffmpeg5.conf >/dev/null
    sudo ldconfig
}

verify_runtime() {
    register_runtime

    if [ ! -x "$PREFIX/bin/ffmpeg" ]; then
        return 1
    fi

    VERSION_OUTPUT="$("$PREFIX/bin/ffmpeg" -version 2>&1)" || return 1
    VERSION_LINE="${VERSION_OUTPUT%%$'\n'*}"

    case "$VERSION_LINE" in
        "ffmpeg version 5.1.9 "*|"ffmpeg version 5.1.9")
            ;;
        *)
            echo "Unexpected FFmpeg version: $VERSION_LINE"
            return 1
            ;;
    esac

    LDD_OUTPUT="$(ldd "$PREFIX/bin/ffmpeg")" || return 1

    if grep -F "not found" <<< "$LDD_OUTPUT" >/dev/null; then
        echo "Unresolved FFmpeg libraries:"
        grep -F "not found" <<< "$LDD_OUTPUT" || true
        return 1
    fi

    return 0
}

if [ -x "$PREFIX/bin/ffmpeg" ]; then
    echo "Existing FFmpeg build detected."

    if verify_runtime; then
        echo "Existing FFmpeg 5.1.9 is healthy; skipping compilation."
        "$PREFIX/bin/ffmpeg" -version
        exit 0
    fi

    echo "Existing FFmpeg is incomplete; rebuilding."
    rm -rf "$PREFIX"
fi

cd /content
rm -rf "$SRC" "$XZ" "$TGZ" /content/FFmpeg-n5.1.9

OFFICIAL_OK=0
echo "Downloading FFmpeg 5.1.9 from ffmpeg.org..."

if curl -L \
    --fail \
    --retry 10 \
    --retry-all-errors \
    --retry-delay 3 \
    --connect-timeout 30 \
    -o "$XZ" \
    https://ffmpeg.org/releases/ffmpeg-5.1.9.tar.xz
then
    if xz -t "$XZ"; then
        OFFICIAL_OK=1
        echo "ffmpeg.org archive integrity: PASS"
    fi
fi

if [ "$OFFICIAL_OK" -eq 1 ]; then
    tar -xf "$XZ"
else
    echo "Using official GitHub n5.1.9 fallback."

    curl -L \
        --fail \
        --retry 10 \
        --retry-all-errors \
        --retry-delay 3 \
        --connect-timeout 30 \
        -o "$TGZ" \
        https://github.com/FFmpeg/FFmpeg/archive/refs/tags/n5.1.9.tar.gz

    gzip -t "$TGZ"
    tar -xzf "$TGZ"

    [ -d /content/FFmpeg-n5.1.9 ] || {
        echo "GitHub source extraction failed."
        exit 1
    }

    mv /content/FFmpeg-n5.1.9 "$SRC"
fi

[ -d "$SRC" ] || {
    echo "FFmpeg source directory missing."
    exit 1
}

cd "$SRC"

./configure \
    --prefix="$PREFIX" \
    --enable-shared \
    --disable-static \
    --disable-doc \
    --disable-debug \
    --enable-gpl \
    --enable-libmp3lame

# Conservative compile parallelism for Colab host RAM.
make -j2
make install

register_runtime
verify_runtime

echo
echo "===================================================="
echo "FFmpeg 5.1.9 setup: PASS"
echo "===================================================="
"$PREFIX/bin/ffmpeg" -version

echo
echo "Resolved FFmpeg libraries:"
LDD_OUTPUT="$(ldd "$PREFIX/bin/ffmpeg")"
grep -E 'libavdevice|libavformat|libavcodec|libavutil|libswresample|libswscale|libavfilter' \
    <<< "$LDD_OUTPUT" || true

# Save disk after a successful build.
rm -rf "$SRC" "$XZ" "$TGZ"
'''

script_path = Path("/content/setup_ffmpeg5.sh")
script_path.write_text(ffmpeg_script, encoding="utf-8")
script_path.chmod(0o755)

process = subprocess.Popen(
    ["stdbuf", "-oL", "-eL", "bash", str(script_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(f"FFmpeg setup failed, return code={rc}")

os.environ["PATH"] = (
    "/content/ffmpeg-5.1.9-build/bin:"
    + os.environ.get("PATH", "")
)
os.environ["LD_LIBRARY_PATH"] = (
    "/content/ffmpeg-5.1.9-build/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print("\nFFmpeg executable:", shutil.which("ffmpeg"))

## 2. 安裝 HeartMuLa 與相依套件

In [ ]:
# Install a matched PyTorch CUDA stack for the official model.
# 安裝版本相容的 PyTorch CUDA 套件。
%pip -q install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128

# Install the audio codec used by the model.
# 安裝模型所需的音訊編解碼套件。
%pip -q install "torchcodec==0.10.0+cu128" --index-url https://download.pytorch.org/whl/cu128

# Install the official inference dependencies without replacing notebook packages.
# 安裝官方推論套件，不改動 Notebook 的核心套件。
%pip -q install numpy==2.0.2 torchtune==0.4.0 torchao==0.9.0 transformers==4.57.0 tokenizers==0.22.1 accelerate==1.12.0 einops==0.8.1 vector-quantize-pytorch==1.27.15 huggingface-hub==0.36.2 hf-xet soundfile tqdm==4.67.1

# Download the official HeartMuLa code.
# 下載 HeartMuLa 官方程式。
!test -d /content/heartlib || git clone -q https://github.com/HeartMuLa/heartlib.git /content/heartlib

# Install only the source code, keeping the dependencies above.
# 只安裝原始碼，沿用上面已安裝的相依套件。
%pip -q install --no-deps -e /content/heartlib

## 3. 匯入模組

In [ ]:
# Import the tools used to prepare and run generation.
# 匯入準備及執行音樂生成所需的工具。
import subprocess
import sys
from pathlib import Path

# Import the official Hugging Face download tool.
# 匯入 Hugging Face 官方模型下載工具。
from huggingface_hub import snapshot_download

# Import audio preview and download tools.
# 匯入音訊播放與下載工具。
from IPython.display import Audio, display
from google.colab import files

## 4. 修改歌詞、曲風及秒數

In [ ]:
# Write the lyrics that the model will sing.
# 設定模型要演唱的歌詞。
LYRICS = """[Verse]
雨後的街道慢慢亮起燈
微風吹過熟悉的旅程
[Chorus]
讓我們迎著光向前走
把每個故事留在心中"""

# Describe the musical style and generation length.
# 設定音樂風格及生成長度。
TAGS = "c-pop, warm, mellow, piano, guitar, medium tempo, female vocal"
DURATION = 20

# Save the settings as text files required by HeartMuLa.
# 將設定寫入 HeartMuLa 需要的文字檔。
Path("/content/my_lyrics.txt").write_text(LYRICS, encoding="utf-8")
Path("/content/my_tags.txt").write_text(TAGS, encoding="utf-8")

## 5. 下載官方模型（首次執行較久）

In [ ]:
# Set the checkpoint directory.
# 設定模型儲存目錄。
CKPT = Path("/content/heartlib/ckpt")

# Download the tokenizer and model configuration.
# 下載 Tokenizer 與模型設定。
snapshot_download("HeartMuLa/HeartMuLaGen", local_dir=str(CKPT), allow_patterns=["tokenizer.json", "gen_config.json"], max_workers=2)

# Download the 3B music generation model.
# 下載 3B 音樂生成模型。
snapshot_download("HeartMuLa/HeartMuLa-oss-3B-happy-new-year", local_dir=str(CKPT / "HeartMuLa-oss-3B"), allow_patterns=["config.json", "*.safetensors", "*.safetensors.index.json"], max_workers=2)

# Download the HeartCodec audio decoder.
# 下載 HeartCodec 音訊解碼器。
snapshot_download("HeartMuLa/HeartCodec-oss-20260123", local_dir=str(CKPT / "HeartCodec-oss"), allow_patterns=["config.json", "*.safetensors", "*.safetensors.index.json"], max_workers=2)
# Confirm that checkpoint files and every indexed weight shard exist.
# 確認模型設定及每個權重分片都已完整下載。
import json
for folder in [CKPT / "HeartMuLa-oss-3B", CKPT / "HeartCodec-oss"]:
    index = folder / "model.safetensors.index.json"
    assert (folder / "config.json").is_file() and index.is_file(), f"Missing config/index: {folder}"
    names = set(json.loads(index.read_text())["weight_map"].values())
    assert names and all((folder / name).is_file() for name in names), f"Missing model shards: {folder}"
print("Model checkpoint verification: PASS")

## 6. 驗證相依性：TorchCodec + FFmpeg 5.1.9

不只查看版本，還會在獨立 Python 程序實際寫出測試 MP3；失敗時停止，不繼續推論。

In [ ]:
# Configure the tested FFmpeg shared-library paths for child processes.
# 設定子程序使用已驗證的 FFmpeg 共用函式庫。
import os
runtime_env = os.environ.copy()
runtime_env["PATH"] = "/content/ffmpeg-5.1.9-build/bin:" + runtime_env.get("PATH", "")
runtime_env["LD_LIBRARY_PATH"] = "/content/ffmpeg-5.1.9-build/lib:" + runtime_env.get("LD_LIBRARY_PATH", "")
runtime_env["PYTHONPATH"] = "/content/heartlib/src:" + runtime_env.get("PYTHONPATH", "")

# Verify audio encoding in a fresh process before running the model.
# 在新程序中實際驗證 MP3 編碼，避免模型生成完卻無法儲存。
check = "import torch,torchaudio; from heartlib import HeartMuLaGenPipeline; torchaudio.save('/content/ffmpeg_test.mp3',torch.zeros(1,4800),48000); print('MP3 encoder OK')"
subprocess.run([sys.executable, "-c", check], env=runtime_env, check=True)
assert Path("/content/ffmpeg_test.mp3").stat().st_size > 0

## 7. 生成音樂

In [ ]:
# Run the official generator with T4 memory-saving options.
# 使用 T4 適用的省記憶體選項執行官方生成器。
command = [
    sys.executable, "/content/heartlib/examples/run_music_generation.py",
    "--model_path", str(CKPT), "--version", "3B",
    "--lyrics", "/content/my_lyrics.txt", "--tags", "/content/my_tags.txt",
    "--save_path", "/content/music_output.mp3",
    "--max_audio_length_ms", str(DURATION * 1000),
    "--lazy_load", "true", "--mula_dtype", "float16", "--codec_dtype", "float32",
]

# Start generation and wait for the MP3 file.
# 開始生成並等待 MP3 檔案完成。
subprocess.run(command, cwd="/content/heartlib", env=runtime_env, check=True)

## 8. 試聽並下載 MP3

In [ ]:
# Play the generated music in the notebook.
# 在 Notebook 裡面播放生成的音樂。
output = Path("/content/music_output.mp3")
display(Audio(filename=str(output)))

# Download the MP3 file to your computer.
# 將 MP3 下載到自己的電腦。
files.download(str(output))